Simple analogy:

Student studied Chapter 1-5 and passed a test

Student already knows Chapter 1-5 well ✅

Just needs to learn Chapter 6-10

Occasional revision of 1-5 keeps memory fresh

If compute was expensive and dataset was huge (like all 798 segments), fine-tuning on new data only would make sense. That's called **incremental learning** and is used in production ML systems.

In ML terms this is called fine-tuning with replay:

For this project — experience replay is the most sophisticated approach and shows deeper ML knowledge in this assignment.

Start from 50-segment trained model(waymo_yolov8m_best.pt)

Train on new 100 segments

Include small % of old 50 segments as "revision" to prevent forgetting

---
So the final plan:

| Step | Action
|-------|-----------|
| 1| Extract segments 51-150 (100 new segments) |
| 2 | Training data = 100 new segments + 10% random sample of first 50 segments |
| 3 | Fine-tune from waymo_yolov8m_best.pt (experience replay) |
| 4 | Compare results with previous 50-segment model|


**Expected training data:**

100 new segments = ~19,800 images

10% of old 50 = ~990 images

Total = ~20,790 images

**Expected improvement:**

mAP@0.5: 0.736 → hopefully ≥0.80

Pedestrian recall: 0.624 → hopefully ≥0.75

Cyclist recall: 0.551 → hopefully ≥0.70


Order of execution:

✅ Cell — os.environ + all imports

✅ Cell — GCS authentication

✅ Cell — tfrecords list + tfrecords_new

✅ Cell — extract_one_segment function definition

✅ Cell — extraction loop

Extract new 100 segments

In [ ]:
# first install packages → restart runtime
!pip install protobuf==3.20.3 -q
!pip install waymo-open-dataset-tf-2-11-0 --no-deps -q
!pip install ultralytics -q
!pip install crcmod -q

print("Installation complete — restart runtime now!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 7.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grain 0.2.16 requires protobuf>=5.28.3, but you have protobuf 3.20.3 which is incompatible.
tensorflow-metadata 1.17.3 requires protobuf>=4.25.2; python_version >= "3.11", but you have protobuf 3.20.3 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 3.20.3 which is incompatible.
google-api-core 2.30.0 requires protobuf<7.0.0,>=4.25.8, but you have protobuf 3.20.3 which is incompatible.
opentelemetry-proto 1.38.0 requires protobuf<7.0,>=5.0, but you have protobuf 3.20.3 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 3.20.3 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━

In [ ]:
#set env variable + all imports (after restart)

import os
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"## Must be FIRST line before any imports

# All imports
import io
import tensorflow as tf
import numpy as np
import pandas as pd
from PIL import Image
from pathlib import Path
import shutil
import random
from ultralytics import YOLO
from waymo_open_dataset import dataset_pb2

print("All imports successful ✅")


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
All imports successful ✅


This line tells Python to use the pure Python implementation of protobuf instead of the C++ implementation.

Because:

Waymo package was built with an older protobuf version. The newer protobuf installed on Colab uses C++ by default which is incompatible:

# Without this line — error:
Descriptors cannot be created directly.

Downgrade protobuf to 3.20.x or lower

OR set PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION=python

Two options to fix — we chose option 2:

| Option | What it does        |
|-------|----------------------|
| Downgrade protobuf| Changes the package version — can break other things |
| Set env variable | Tells protobuf to use Python mode — slower but compatible |


The environment variable must be set before from waymo_open_dataset import dataset_pb2 — once protobuf loads it can't be changed mid-session.

**Performance impact:**

Pure Python protobuf is slower than C++ — but only affects TFRecord parsing (extraction step), not YOLOv8 training. So no impact on training speed.


In [ ]:
#authenticate GCS
!gcloud auth login --no-launch-browser


Go to the following link in your browser, and complete the sign-in prompts:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=32555940559.apps.googleusercontent.com&redirect_uri=https%3A%2F%2Fsdk.cloud.google.com%2Fauthcode.html&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fappengine.admin+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcompute+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Faccounts.reauth&state=iRrcNqbJNA9c2KyXD0xLop1cL71NU0&prompt=consent&token_usage=remote&access_type=offline&code_challenge=upkX8cYZ8bTKbbO7cK11o-TvDmzgjq0MsfyaCr855f0&code_challenge_method=S256

Once finished, enter the verification code provided in your browser: 4/0AfrIepD0ETffdPllxTWdFdYPTjvYwyUfMWXQP9vcGoQ-pfVuuAAaFRpHoKRBeChTtvrWag

You are now logged in as [suh2162674@maricopa.edu].
Your current projec

In [ ]:
#Enable L4 GPU in colab
# Check environment
import os
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

# Check GCS access
!gsutil ls gs://mywaymo-perdataset-2026/
print("GCS connected ✅")

NVIDIA L4, 23034 MiB
gs://mywaymo-perdataset-2026/samples.npy
gs://mywaymo-perdataset-2026/extracted/
gs://mywaymo-perdataset-2026/models/
gs://mywaymo-perdataset-2026/training/
GCS connected ✅


In [ ]:
# Get all 798 TFRecords
result = !gsutil ls gs://mywaymo-perdataset-2026/training/
tfrecords = [r.strip() for r in result if '.tfrecord' in r]
print(f"Total TFRecords available: {len(tfrecords)}")

# Extract segments 51-150
tfrecords_new = tfrecords[50:150]
print(f"Segments to extract: {len(tfrecords_new)}")

Total TFRecords available: 798
Segments to extract: 100


In [ ]:
#Extraction function for segments 51-150
from pathlib import Path
import os

LOCAL_WORK = Path("/content/extraction")
LOCAL_WORK.mkdir(exist_ok=True)

GCS_EXTRACTED = "gs://mywaymo-perdataset-2026/extracted"

def extract_one_segment(tfrecord_path):

    seg_name = tfrecord_path.split('/')[-1].replace('.tfrecord-00000-of-00001', '').replace('.tfrecord', '')
    local_dir = f"/content/extracted/{seg_name}"
    os.makedirs(local_dir, exist_ok=True)

    print(f"  Step 1: Downloading TFRecord...")
    url = tfrecord_path.strip()
    !gsutil -q cp {url} /content/current.tfrecord

    print(f"  Step 2: Parsing frames...")
    dataset = tf.data.TFRecordDataset("/content/current.tfrecord")
    images_saved = 0
    labels_list = []

    for frame_idx, data in enumerate(dataset):
        frame = dataset_pb2.Frame()
        frame.ParseFromString(data.numpy())

        for cam in frame.images:
            if cam.name != 1:
                continue
            img = Image.open(io.BytesIO(cam.image))
            img.save(f"{local_dir}/frame_{frame_idx:03d}_FRONT.jpg")
            images_saved += 1

            for cam_labels in frame.camera_labels:
                if cam_labels.name != 1:
                    continue
                for label in cam_labels.labels:
                    labels_list.append({
                        'frame': frame_idx,
                        'camera': 'FRONT',
                        'type': label.type,
                        'center_x': label.box.center_x,
                        'center_y': label.box.center_y,
                        'width': label.box.width,
                        'height': label.box.length
                    })

    print(f"  Step 3: Saving labels...")
    df = pd.DataFrame(labels_list)
    df.to_csv(f"{local_dir}/labels.csv", index=False)

    print(f"  Step 4: Uploading to GCS...")
    gcs_path = f"{GCS_EXTRACTED}/{seg_name}"
    local_jpg = f"{local_dir}/*.jpg"
    local_csv = f"{local_dir}/labels.csv"
    !gsutil -m -q cp {local_jpg} {gcs_path}/
    !gsutil -q cp {local_csv} {gcs_path}/labels.csv

    import shutil
    shutil.rmtree(local_dir)
    os.remove("/content/current.tfrecord")

    print(f"  ✅ Done! {images_saved} images, {len(df)} labels")

print("Function ready ✅")

Function ready ✅


In [ ]:
#extraction loop:
# Extract 50-150 segments one by one
total = len(tfrecords_new)

for i, tfrecord_path in enumerate(tfrecords_new):
    seg_name = tfrecord_path.split('/')[-1].replace('.tfrecord-00000-of-00001', '').replace('.tfrecord', '')

    # Skip if already extracted
    check = !gsutil ls {GCS_EXTRACTED}/{seg_name}/labels.csv 2>/dev/null
    if check:
        print(f"Segment {i+1}/{total} — already extracted ✅")
        continue

    print(f"\nSegment {i+1}/{total}")
    try:
        extract_one_segment(tfrecord_path)
    except Exception as e:
        print(f"  ❌ Failed: {e} — skipping")
        continue

print("\n🎉 All 100 new segments extracted!")


Segment 1/100
  Step 1: Downloading TFRecord...
  Step 2: Parsing frames...
  Step 3: Saving labels...
  Step 4: Uploading to GCS...
  ✅ Done! 197 images, 663 labels

Segment 2/100
  Step 1: Downloading TFRecord...
  Step 2: Parsing frames...
  Step 3: Saving labels...
  Step 4: Uploading to GCS...
  ✅ Done! 189 images, 1079 labels

Segment 3/100
  Step 1: Downloading TFRecord...
  Step 2: Parsing frames...
  Step 3: Saving labels...
  Step 4: Uploading to GCS...
  ✅ Done! 199 images, 370 labels

Segment 4/100
  Step 1: Downloading TFRecord...
  Step 2: Parsing frames...
  Step 3: Saving labels...
  Step 4: Uploading to GCS...
  ✅ Done! 197 images, 1773 labels

Segment 5/100
  Step 1: Downloading TFRecord...
  Step 2: Parsing frames...
  Step 3: Saving labels...
  Step 4: Uploading to GCS...
  ✅ Done! 199 images, 1205 labels

Segment 6/100
  Step 1: Downloading TFRecord...
  Step 2: Parsing frames...
  Step 3: Saving labels...
  Step 4: Uploading to GCS...
  ✅ Done! 198 images, 12754 

In [ ]:
#should see 150 segments total (50 old + 100 new).
#Expected data:150 × 198 = ~29,700 images

result = !gsutil ls gs://mywaymo-perdataset-2026/extracted/
segments = [r.strip() for r in result if 'segment' in r]
print(f"Total segments in GCS: {len(segments)}")
!gsutil du -sh gs://mywaymo-perdataset-2026/extracted/

Total segments in GCS: 150
6.26 GiB     gs://mywaymo-perdataset-2026/extracted


**Training pipeline.**

In [ ]:
#runevery time after session break
from pathlib import Path

GCS_EXTRACTED  = "gs://mywaymo-perdataset-2026/extracted"
GCS_MODEL_OUT  = "gs://mywaymo-perdataset-2026/models"
LOCAL_DATA_DIR = Path("/content/waymo_yolo")
LOCAL_DATA_DIR.mkdir(exist_ok=True)

CLASS_MAP    = {1: 0, 2: 1, 3: 2, 4: 3}
CLASS_NAMES  = ["Vehicle", "Pedestrian", "Sign", "Cyclist"]
IMG_W, IMG_H = 1920, 1280
CAMERA       = "FRONT"
TRAIN_SPLIT  = 0.8
EPOCHS       = 50
IMG_SIZE     = 640
BATCH_SIZE   = 32
OLD_SEGMENTS = 50   # first 50 segments
NEW_SEGMENTS = 100  # new 100 segments
REPLAY_PCT   = 0.1  # 10% of old segments for experience replay

print("Config ready ✅")
print(f"Old segments : {OLD_SEGMENTS}")
print(f"New segments : {NEW_SEGMENTS}")
print(f"Replay %     : {int(REPLAY_PCT*100)}% of old segments")

Config ready ✅
Old segments : 50
New segments : 100
Replay %     : 10% of old segments


In [ ]:
# Data Download

# Get segment lists from GCS
result = !gsutil ls {GCS_EXTRACTED}/
all_segments = [r.strip().rstrip('/') for r in result if 'segment' in r]

# Split into old and new
old_segments = all_segments[:50]   # first 50
new_segments = all_segments[50:]   # next 100

print(f"Old segments : {len(old_segments)}")
print(f"New segments : {len(new_segments)}")

# Experience replay — sample 10% of old segments
import random
random.seed(42)
replay_segments = random.sample(old_segments, int(len(old_segments) * REPLAY_PCT))
print(f"Replay segments selected : {len(replay_segments)}")

# Final training segments = new 100 + 10% old
training_segments = new_segments + replay_segments
print(f"Total training segments  : {len(training_segments)}")

Old segments : 50
New segments : 100
Replay segments selected : 5
Total training segments  : 105




This line:

```python
img.rename(raw_images_dir / new_name)
```

`rename()` in Python's pathlib **moves** the file from `temp_dir` to `raw_images_dir` with the new prefixed name.

So the flow is:
1. Download to `temp_dir` — `gsutil -m cp`
2. Rename + move to `raw_images_dir` — `img.rename()`
3. Delete `temp_dir` — `shutil.rmtree()`

All images end up in `raw_images_dir` with unique segment-prefixed names.

In [ ]:
#Download Images and labels

# Setup directories
raw_images_dir = LOCAL_DATA_DIR / "raw_images"
raw_labels_dir = LOCAL_DATA_DIR / "raw_labels"
raw_images_dir.mkdir(exist_ok=True)
raw_labels_dir.mkdir(exist_ok=True)

all_labels = []

for i, seg_path in enumerate(training_segments):
    seg_name = seg_path.split('/')[-1]
    seg_short = f"seg{i:03d}"
    print(f"Downloading segment {i+1}/{len(training_segments)}: {seg_name[:40]}...")

    # Download to temp folder first
    temp_dir = LOCAL_DATA_DIR / "temp"
    temp_dir.mkdir(exist_ok=True)
    !gsutil -m -q cp "{seg_path}/*_FRONT.jpg" "{temp_dir}/"

    # Rename with segment prefix and move
    for img in temp_dir.glob("*.jpg"):
        new_name = f"{seg_short}_{img.name}"
        img.rename(raw_images_dir / new_name)

    shutil.rmtree(str(temp_dir))

    # Download labels
    csv_local = raw_labels_dir / f"{seg_name}_labels.csv"
    !gsutil -q cp "{seg_path}/labels.csv" "{csv_local}"

    if csv_local.exists():
        df = pd.read_csv(csv_local)
        df['segment'] = seg_name
        df['seg_short'] = seg_short
        all_labels.append(df)

# Combine all labels
labels_df = pd.concat(all_labels, ignore_index=True)
labels_df = labels_df[labels_df['camera'] == CAMERA].reset_index(drop=True)

print(f"\nTotal images : {len(list(raw_images_dir.glob('*.jpg')))}")
print(f"Total labels : {len(labels_df)}")


Total images : 20785
Total labels : 518380


|       |Expected  |Actual |
|-------|----------|-------|
|Segments|105      |  105  |
|Images|105 × 198 = ~20,790|20,785 ✅|
|Labels|105 × ~4,500 avg|518,380 ✅|

Images are slightly less than expected — a few frames had no FRONT camera images, completely normal.

Labels jumped from 224,023 (50 segments) to 518,380 (105 segments) — more complex urban scenes in new segments with more objects per frame.

In [ ]:
#run label conversion:
# Create YOLO label files
yolo_labels_dir = LOCAL_DATA_DIR / "yolo_labels"
yolo_labels_dir.mkdir(exist_ok=True)

def get_image_size(img_path):
    try:
        with Image.open(img_path) as img:
            return img.size
    except:
        return IMG_W, IMG_H

def convert_to_yolo(center_x, center_y, width, height, img_w, img_h):
    cx = max(0.0, min(1.0, center_x / img_w))
    cy = max(0.0, min(1.0, center_y / img_h))
    w  = max(0.001, min(1.0, width / img_w))
    h  = max(0.001, min(1.0, height / img_h))
    return cx, cy, w, h

skipped = 0
processed = 0

for (seg_short, frame), group in labels_df.groupby(['seg_short', 'frame']):
    img_filename = f"{seg_short}_frame_{int(frame):03d}_FRONT.jpg"
    img_path = raw_images_dir / img_filename

    if not img_path.exists():
        skipped += 1
        continue

    img_w, img_h = get_image_size(img_path)
    label_path = yolo_labels_dir / img_filename.replace('.jpg', '.txt')

    with open(label_path, 'w') as f:
        for _, row in group.iterrows():
            yolo_class = CLASS_MAP.get(int(row['type']))
            if yolo_class is None:
                continue
            cx, cy, w, h = convert_to_yolo(
                row['center_x'], row['center_y'],
                row['width'], row['height'],
                img_w, img_h
            )
            f.write(f"{yolo_class} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}\n")

    processed += 1

print(f"YOLO labels created : {processed}")
print(f"Skipped             : {skipped}")

YOLO labels created : 20066
Skipped             : 0


The difference between 20,785 images and 20,066 label files means ~719 frames had no objects — empty scenes. Completely normal.

In [ ]:
#train/val split

# Create train/val split
images_dir = LOCAL_DATA_DIR / "images"
labels_dir = LOCAL_DATA_DIR / "labels"

(images_dir / "train").mkdir(parents=True, exist_ok=True)
(images_dir / "val").mkdir(parents=True, exist_ok=True)
(labels_dir / "train").mkdir(parents=True, exist_ok=True)
(labels_dir / "val").mkdir(parents=True, exist_ok=True)

# Only use images that have labels
labeled_images = [f for f in raw_images_dir.glob("*.jpg")
                  if (yolo_labels_dir / f.name.replace('.jpg', '.txt')).exists()]

# Shuffle and split
random.seed(42)
random.shuffle(labeled_images)
split_idx = int(len(labeled_images) * TRAIN_SPLIT)
train_imgs = labeled_images[:split_idx]
val_imgs   = labeled_images[split_idx:]

# Copy to train/val folders
for img in train_imgs:
    shutil.copy(img, images_dir / "train" / img.name)
    shutil.copy(yolo_labels_dir / img.name.replace('.jpg', '.txt'),
                labels_dir / "train" / img.name.replace('.jpg', '.txt'))

for img in val_imgs:
    shutil.copy(img, images_dir / "val" / img.name)
    shutil.copy(yolo_labels_dir / img.name.replace('.jpg', '.txt'),
                labels_dir / "val" / img.name.replace('.jpg', '.txt'))

print(f"Train images : {len(train_imgs)}")
print(f"Val images   : {len(val_imgs)}")

Train images : 16052
Val images   : 4014


In [ ]:
# Check what models are available
!gsutil ls gs://mywaymo-perdataset-2026/models/

gs://mywaymo-perdataset-2026/models/BoxF1_curve.png
gs://mywaymo-perdataset-2026/models/BoxPR_curve.png
gs://mywaymo-perdataset-2026/models/BoxP_curve.png
gs://mywaymo-perdataset-2026/models/BoxR_curve.png
gs://mywaymo-perdataset-2026/models/confusion_matrix.png
gs://mywaymo-perdataset-2026/models/confusion_matrix_normalized.png
gs://mywaymo-perdataset-2026/models/results.png
gs://mywaymo-perdataset-2026/models/waymo_yolov8m_best.pt
gs://mywaymo-perdataset-2026/models/waymo_yolov8m_last.pt
gs://mywaymo-perdataset-2026/models/waymo_yolov8n_best.pt
gs://mywaymo-perdataset-2026/models/waymo_yolov8n_last.pt


## Experiment Plan

### Completed
| Exp | Model | Data | mAP@0.5 | mAP@0.5:0.95 | Inference | Status |
|-----|-------|------|---------|--------------|-----------|--------|
| Baseline | YOLOv8n | 2 segments | 0.461 | 0.274 | 18.8ms | ✅ Done |
| Exp 1 | YOLOv8m | 50 segments | 0.736 | 0.478 | 4.8ms | ✅ Done |

### In Progress
| Exp | Model | Data | Approach |
|-----|-------|------|---------|
| Exp 2 | YOLOv8m | 105 segments | Experience replay from Exp 1 weights |

## Updated Experiment Plan

| Exp | Model | Data | mAP@0.5 | Status |
|-----|-------|------|---------|--------|
| Exp 1 | YOLOv8m | 50 segments | 0.736 | ✅ Done |
| Exp 2 | YOLOv8m Experience Replay | 105 segments | 0.687 | ✅ Done |
| Exp 3 | YOLO26x | 150 segments | TBD | 🔲 Next |
| Exp 4 | RF-DETR-L | 150 segments | TBD | 🔲 Planned |
| Exp 5 | RF-DETR-L + YOLO26x Ensemble | 150 segments | TBD | 🔲 Planned |

### Key Learnings So Far
- Exp 1 → Exp 2: Experience replay with 10% old data
  insufficient — mAP dropped from 0.736 to 0.687
- More data alone does not guarantee improvement —
  data distribution shift matters
- Next experiments use full 150 segments from scratch
  with stronger architectures
### Production Selection Criteria
| Criteria | Target | Current Best |
|----------|--------|-------------|
| mAP@0.5 | ≥ 0.80 | 0.736 |
| mAP@0.5:0.95 | ≥ 0.50 | 0.478 |
| Pedestrian recall | ≥ 0.90 | 0.624 |
| Cyclist recall | ≥ 0.95 | 0.551 |
| Inference speed | ≤ 50ms | 4.8ms |

#### Best model selected for production after all experiments complete.

### Reference
Roboflow Model Leaderboard: https://leaderboard.roboflow.com

In [ ]:
# Create YAML;
#waymo.yaml — the config file that tells YOLOv8 where to find training & validation images,how many classes, and class names.
from pathlib import Path

LOCAL_DATA_DIR = Path("/content/waymo_yolo")
CLASS_NAMES = ["Vehicle", "Pedestrian", "Sign", "Cyclist"]

yaml_content = f"""
path: {LOCAL_DATA_DIR}
train: images/train
val: images/val

nc: 4
names: {CLASS_NAMES}
"""

yaml_path = LOCAL_DATA_DIR / "waymo.yaml"
with open(yaml_path, 'w') as f:
    f.write(yaml_content)

print(f"YAML created ✅")
print(yaml_content)

YAML created ✅

path: /content/waymo_yolo
train: images/train
val: images/val

nc: 4
names: ['Vehicle', 'Pedestrian', 'Sign', 'Cyclist']



In [ ]:
#Save to GCS
# Save prepared dataset to GCS — run ONCE after dataset ready
print("Saving dataset to GCS...")
!gsutil -m -q cp -r /content/waymo_yolo/images gs://mywaymo-perdataset-2026/prepared/
!gsutil -m -q cp -r /content/waymo_yolo/labels gs://mywaymo-perdataset-2026/prepared/
!gsutil -q cp /content/waymo_yolo/waymo.yaml gs://mywaymo-perdataset-2026/prepared/
print("Dataset saved to GCS ✅")

Saving dataset to GCS...
Dataset saved to GCS ✅


In [ ]:
#Download trained model from gcp:
!gsutil cp gs://mywaymo-perdataset-2026/models/waymo_yolov8m_best.pt /content/waymo_yolov8m_best.pt
MODEL = "/content/waymo_yolov8m_best.pt"
print(f"Model ready: {MODEL} ✅")

Copying gs://mywaymo-perdataset-2026/models/waymo_yolov8m_best.pt...
- [1 files][ 49.6 MiB/ 49.6 MiB]                                                
Operation completed over 1 objects/49.6 MiB.                                     
Model ready: /content/waymo_yolov8m_best.pt ✅


In [ ]:
# Train — Experience Replay
from ultralytics import YOLO

model = YOLO(MODEL)

results = model.train(
    data=str(yaml_path),
    epochs=50,
    imgsz=640,
    batch=32,
    name="waymo_yolov8m_replay",
    project="/content/runs",
    device=0,
    workers=4,
    patience=10,
    save=True,
    plots=True,
    verbose=True
)

print("Training complete ✅")

Ultralytics 8.4.21 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/waymo_yolo/waymo.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/waymo_yolov8m_best.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=waymo_yolov8m_replay, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_ma

JavaScript keep-alive only stays active for the current browser session. When you close or refresh the browser it stops.
Every new session you need to run it again:

Press Cmd + Option + J in Chrome

Type allow pasting → Enter

Paste the JavaScript → Enter

Keep Colab open, run the JavaScript keep-alive in browser console

function KeepAlive() {

  document.querySelector("#top-toolbar").click();

  console.log("Keeping session alive...");
}
setInterval(KeepAlive, 60000);

In [ ]:
#Check if GPU is still allocated:
!nvidia-smi

Fri Mar  6 22:14:39 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   39C    P8             13W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
#Resume from epoch 11 instead of starting over:
model = YOLO("/content/runs/waymo_yolov8m_replay4/weights/last.pt")

results = model.train(
    resume=True
)

print("Training resumed ✅")

In [ ]:
# Training complete! Save model to GCS

!gsutil cp /content/runs/waymo_yolov8m_replay/weights/best.pt gs://mywaymo-perdataset-2026/models/waymo_yolov8m_replay_best.pt
!gsutil cp /content/runs/waymo_yolov8m_replay/weights/last.pt gs://mywaymo-perdataset-2026/models/waymo_yolov8m_replay_last.pt
!gsutil -m -q cp /content/runs/waymo_yolov8m_replay/*.png gs://mywaymo-perdataset-2026/models/replay/
print("Model saved ✅")


Copying file:///content/runs/waymo_yolov8m_replay/weights/best.pt [Content-Type=application/vnd.snesdev-page-table]...
-
Operation completed over 1 objects/49.6 MiB.                                     
Copying file:///content/runs/waymo_yolov8m_replay/weights/last.pt [Content-Type=application/vnd.snesdev-page-table]...
-
Operation completed over 1 objects/49.6 MiB.                                     
Model saved ✅


## Exp 2 Results — YOLOv8m Experience Replay (105 segments)

### Final Validation Metrics

| Class | Images | Instances | Precision | Recall | mAP@0.5 | mAP@0.5:0.95 |
|-------|--------|-----------|-----------|--------|---------|--------------|
| All | 4014 | 103772 | 0.857 | 0.607 | 0.687 | 0.433 |
| Vehicle | 3948 | 77234 | 0.899 | 0.703 | 0.781 | 0.556 |
| Pedestrian | 2289 | 25791 | 0.866 | 0.604 | 0.698 | 0.413 |
| Cyclist | 594 | 747 | 0.807 | 0.515 | 0.582 | 0.330 |

### Comparison vs Exp 1

| Metric | Exp 1 (50 seg) | Exp 2 (105 seg replay) | Change |
|--------|---------------|----------------------|--------|
| mAP@0.5 | 0.736 | 0.687 | -0.049 ⚠️ |
| mAP@0.5:0.95 | 0.478 | 0.433 | -0.045 ⚠️ |
| Precision | 0.925 | 0.857 | -0.068 ⚠️ |
| Recall | 0.630 | 0.607 | -0.023 ⚠️ |
| Pedestrian Recall | 0.624 | 0.604 | -0.020 ⚠️ |
| Cyclist Recall | 0.551 | 0.515 | -0.036 ⚠️ |
| Inference Speed | 4.8ms | 3.8ms | -1.0ms ✅ |

### Conclusion
Experience replay with 10% old segments (5 of 50) was
insufficient to prevent catastrophic forgetting. New 100
segments have different scene distributions — 5 replay
segments not enough to preserve old knowledge.

Next step: Train on all 150 segments combined from scratch
(Exp 3) to recover and exceed Exp 1 performance.

In [ ]:
# Restore dataset from GCS (future sessions)
#after config cell run,noneed to run-> Segment selection,Download images,Label conversion,Train/val split
#continue to run YAML cell,Download model,Training
from pathlib import Path
LOCAL_DATA_DIR = Path("/content/waymo_yolo")
LOCAL_DATA_DIR.mkdir(exist_ok=True)

!gsutil -m -q cp -r gs://mywaymo-perdataset-2026/prepared/images /content/waymo_yolo/
!gsutil -m -q cp -r gs://mywaymo-perdataset-2026/prepared/labels /content/waymo_yolo/
!gsutil -q cp gs://mywaymo-perdataset-2026/prepared/waymo.yaml /content/waymo_yolo/
print("Dataset restored ✅")

In [ ]:
import os
print("waymo_yolo exists:", os.path.exists("/content/waymo_yolo"))
print("model exists:", os.path.exists("/content/waymo_yolov8m_best.pt"))
print("images exist:", os.path.exists("/content/waymo_yolo/images/train"))

waymo_yolo exists: False
model exists: False
images exist: False
